<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/NEMO_GGUF_BENCH_ROBOTIC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## SETUP

In [1]:
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu130 \
  --no-cache-dir

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu130
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.8/782.8 MB 38.9 MB/s  0:00:19
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [llama-cpp-python]


In [2]:
from llama_cpp import llama_supports_gpu_offload

# This will return True if compiled with CUDA/Metal support
print("GPU Offload Supported:", llama_supports_gpu_offload())

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 22563 MiB):
  Device 0: NVIDIA L4, compute capability 8.9, VMM: yes, VRAM: 22563 MiB


GPU Offload Supported: True


In [3]:
!nvidia-smi

Tue Nov 10 12:10:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   53C    P8             14W /   72W |       3MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## CASE1

In [1]:
# 1. Install nvidia-ml-py for live hardware metrics tracking
!pip install -q nvidia-ml-py

# 2. Import libraries and initialize NVIDIA NVML for hardware profiling
from google.colab import drive
from llama_cpp import Llama
import pynvml
import time

drive.mount('/content/drive', force_remount=False)

# Initialize NVML to query L4 GPU metrics
pynvml.nvmlInit()
handle = pynvml.nvmlDeviceGetHandleByIndex(0) # Index 0 for Colab L4 GPU
gpu_name = pynvml.nvmlDeviceGetName(handle)
if isinstance(gpu_name, bytes):
    gpu_name = gpu_name.decode('utf-8')
print(f"Hardware Detected: {gpu_name}\n" + "-"*50)

def print_hardware_metrics(stage_name="Current Status"):
    mem_info = pynvml.nvmlDeviceGetMemoryInfo(handle)
    util = pynvml.nvmlDeviceGetUtilizationRates(handle)
    temp = pynvml.nvmlDeviceGetTemperature(handle, pynvml.NVML_TEMPERATURE_GPU)

    print(f"[{stage_name} Hardware Telemetry]")
    print(f"  • GPU Utilization      : {util.gpu}%")
    print(f"  • VRAM Used / Total    : {mem_info.used / (1024**3):.2f} GB / {mem_info.total / (1024**3):.2f} GB")
    print(f"  • VRAM Usage %         : {(mem_info.used / mem_info.total) * 100:.1f}%")
    print(f"  • GPU Temperature      : {temp}°C")
    print("-" * 50)

# 3. Baseline hardware state before model load
print_hardware_metrics("Before Model Load")

# 4. Set model path and initialize with mixed CPU/GPU offloading
model_path = "/content/drive/MyDrive/GGUF_Models/nemotron-3.5-bolt-topo-governed-Q4_K_M.gguf"

llm = Llama(
    model_path=model_path,
    n_gpu_layers=30,      # Offload 30 layers to GPU, remainder on CPU
    n_ctx=8192,
    n_batch=512,
    verbose=False
)

# Hardware state after model is loaded into mixed CPU/VRAM
print_hardware_metrics("After Model Load (30 Layers on L4 GPU)")

# 5. Run inference test with active metrics profiling
prompt_text = "Analyze the following telemetry input: CPU utilization nominal. System operating normally."
print(f"Prompt: {prompt_text}\n")

start_time = time.time()
output = llm(
    prompt_text,
    max_tokens=150,
    temperature=0.7
)
elapsed_time = time.time() - start_time

print("Model Output:")
print(output['choices'][0]['text'].strip())
print("\n" + "="*50)
print(f"Generation Time: {elapsed_time:.2f} seconds")
print(f"Tokens Generated: {output['usage']['completion_tokens']}")
print(f"Speed: {output['usage']['completion_tokens'] / elapsed_time:.2f} tokens/sec")

# Hardware state post-inference
print_hardware_metrics("Post-Inference State")

# Shutdown NVML handle
pynvml.nvmlShutdown()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Hardware Detected: NVIDIA L4
--------------------------------------------------
[Before Model Load Hardware Telemetry]
  • GPU Utilization      : 0%
  • VRAM Used / Total    : 0.46 GB / 22.49 GB
  • VRAM Usage %         : 2.0%
  • GPU Temperature      : 69°C
--------------------------------------------------


llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 1024
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 1024


[After Model Load (30 Layers on L4 GPU) Hardware Telemetry]
  • GPU Utilization      : 0%
  • VRAM Used / Total    : 3.50 GB / 22.49 GB
  • VRAM Usage %         : 15.6%
  • GPU Temperature      : 70°C
--------------------------------------------------
Prompt: Analyze the following telemetry input: CPU utilization nominal. System operating normally.

Model Output:
No critical alerts detected. Memory usage within acceptable thresholds. Disk I/O stable.

**Analysis:**

The telemetry input describes a system operating in a **normal, healthy state**.

*   **CPU Utilization:** Nominal (Normal operation).
*   **System Status:** Operating normally.
*   **Alerts:** No critical alerts detected.
*   **Memory Usage:** Within acceptable thresholds.
*   **Disk I/O:** Stable.

**Conclusion:**
Based on all provided metrics, there is **no immediate concern or required action** from an operational standpoint. The system is stable, stable, and functioning within established operational parameters.

**(Se

## CASE2

In [1]:
# 1. Mount Google Drive and initialize telemetry and model libraries
from google.colab import drive
from llama_cpp import Llama
import pynvml
import time
import json

drive.mount('/content/drive', force_remount=False)

SEED=123

# Initialize NVIDIA NVML for real-time hardware profiling during simulation
pynvml.nvmlInit()
handle = pynvml.nvmlDeviceGetHandleByIndex(0)
gpu_name = pynvml.nvmlDeviceGetName(handle)
if isinstance(gpu_name, bytes):
    gpu_name = gpu_name.decode('utf-8')

print(f"Simulated Robotics Node Initialized on: {gpu_name}\n" + "-"*50)

def log_telemetry(step_name):
    mem = pynvml.nvmlDeviceGetMemoryInfo(handle)
    util = pynvml.nvmlDeviceGetUtilizationRates(handle)
    temp = pynvml.nvmlDeviceGetTemperature(handle, pynvml.NVML_TEMPERATURE_GPU)
    print(f"[Telemetry | {step_name}]")
    print(f"  • GPU Load : {util.gpu}% | VRAM: {mem.used / (1024**3):.2f} GB / {mem.total / (1024**3):.2f} GB ({(mem.used/mem.total)*100:.1f}%) | Temp: {temp}°C")
    print("-" * 50)

# 2. Load the Model from Drive with Mixed L4 GPU Offloading
model_path = "/content/drive/MyDrive/GGUF_Models/nemotron-3.5-bolt-topo-governed-Q4_K_M.gguf"

llm = Llama(
    model_path=model_path,
    n_gpu_layers=30,      # Offload 30 transformer layers to L4 GPU
    n_ctx=8192,
    n_batch=512,
    verbose=False
)

log_telemetry("Post Model Load")

# 3. Define Simulated Robotics Sensor Stream
simulated_sensor_feed = {
    "timestamp": "2026-11-10T07:25:00Z",
    "rover_id": "SK-ROVER-04",
    "subsystems": {
        "lidar_array": {"status": "nominal", "point_cloud_latency_ms": 14.2},
        "thermal_manifold": {"core_temp_c": 78.4, "warning": "approaching upper threshold"},
        "motor_actuators": {"left_track_rpm": 120.5, "right_track_rpm": 118.9, "slip_detected": False},
        "power_bus": {"voltage_v": 48.2, "draw_amps": 12.4}
    }
}

prompt = f"""[ROBOTICS COGNITIVE CONTROLLER]
Analyze the following real-time sensor telemetry package, identify any anomalies or thermal risks, and output a structured operational directive (Maintain, Throttle, or Emergency Stop) with a brief justification.

Sensor Data:
{json.dumps(simulated_sensor_feed, indent=2)}
"""

# 4. Execute Inference and Measure Performance
start_time = time.time()

output = llm(
    prompt,
    max_tokens=250,
    temperature=0.3,
    seed=SEED,           # Locks down reproducibility
    stop=["<|eot_id|>"]
)

elapsed = time.time() - start_time

print("--- ROBOTIC COGNITIVE RESPONSE ---")
print(output['choices'][0]['text'].strip())
print("\n" + "="*50)
print(f"Inference Latency: {elapsed:.2f} seconds")
print(f"Tokens Generated: {output['usage']['completion_tokens']}")
print(f"Speed: {output['usage']['completion_tokens'] / elapsed:.2f} tokens/sec")
print(f"Seed: {SEED}")
print("\n" + "="*50)
print(f"Processing Speed: {output['usage']['completion_tokens'] / elapsed:.2f} tokens/sec")

log_telemetry("Post Simulation State")
pynvml.nvmlShutdown()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Simulated Robotics Node Initialized on: NVIDIA L4
--------------------------------------------------


llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 1024
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 1024


[Telemetry | Post Model Load]
  • GPU Load : 0% | VRAM: 3.50 GB / 22.49 GB (15.6%) | Temp: 74°C
--------------------------------------------------
--- ROBOTIC COGNITIVE RESPONSE ---
***
**ROBOTICS COGNITIVE CONTROLLER OUTPUT:**
**Directive:** Throttle
**Justification:** Core temperature (78.4°C) is elevated and approaching the upper threshold. Reducing operational load (throttling) will allow the thermal management system to stabilize the core temperature without immediate shutdown. Monitor temperature closely for escalation.

***
**ANALYSIS COMPLETE.**

Inference Latency: 3.87 seconds
Tokens Generated: 79
Speed: 20.43 tokens/sec
Seed: 123

Processing Speed: 20.43 tokens/sec
[Telemetry | Post Simulation State]
  • GPU Load : 21% | VRAM: 3.56 GB / 22.49 GB (15.8%) | Temp: 76°C
--------------------------------------------------


In [2]:
!nvidia-smi

Tue Nov 10 12:34:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   75C    P0             33W /   72W |    3170MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----